# Temă de Programare: Analiza Tiparelor de Valori Lipsă

Bun venit la tema **Analiza Tiparelor de Valori Lipsa**!

Numararea valorilor lipsa iti spune *cat de multe* date lipsesc; intelegerea motivului pentru care lipsesc iti permite sa alegi strategii valide de tratare. Asa cum explica teoria, Donald Rubin (1976) clasifica lipsa datelor in trei mecanisme:

- **MCAR** — lipsa este complet aleatoare, independenta de toate variabilele
- **MAR** — lipsa depinde de variabile *observate*, dar nu de valoarea lipsa in sine
- **MNAR** — lipsa depinde de valoarea lipsa in sine (cel mai periculos si cel mai greu de detectat)

**Ce vei face in aceasta tema:**

* Vei crea variabile indicator binare de lipsa pentru fiecare coloana
* Vei implementa un test MCAR simplificat folosind comparatii statistice intre grupuri definite prin lipsa
* Vei testa tipare MAR prin examinarea relatiilor dintre indicatori si valorile observate
* Vei calcula corelatii perechi intre tiparele de lipsa ale coloanelor
* Vei construi un clasificator euristic care identifica tipul probabil de lipsa pentru o coloana

Sa incepem!


---
<a name='submission'></a>

<h4 style="color:green; font-weight:bold;">SFATURI PENTRU EVALUAREA CU SUCCES A TEMEI TALE:</h4>

* Toate celulele sunt blocate cu excepția celor în care trebuie să trimiți soluțiile sau atunci când este menționat explicit că poți interacționa cu ele.

* În fiecare celulă de exercițiu, caută comentariile `### ÎNCEPUT DE COD AICI ###` și `### SFÂRȘIT DE COD AICI ###`. Acestea îți arată unde să scrii codul soluției. **Nu adăuga sau modifica niciun cod care se află în afara acestor comentarii**.

* Poți adăuga celule noi pentru a experimenta, dar acestea vor fi omise de evaluator, deci nu te baza pe celule create nou pentru a găzdui codul soluției — folosește locurile prevăzute în acest scop.

---

## Cuprins
- [Importuri](#imports)
- [1 - Intelegerea Tiparelor de Valori Lipsa](#1---understanding-missingness-patterns)
    - [1.1 - Generarea de date exemplu cu mecanisme cunoscute](#11---generating-sample-data)
    - **[Exercitiul 1 - `create_missingness_indicators`](#exercise-1---create_missingness_indicators)**
- [2 - Testarea pentru MCAR](#2---testing-for-mcar)
    - [2.1 - Logica din spatele testarii MCAR](#21---the-logic-behind-mcar-testing)
    - **[Exercitiul 2 - `simplified_mcar_test`](#exercise-2---simplified_mcar_test)**
- [3 - Testarea pentru MAR](#3---testing-for-mar)
    - [3.1 - Intelegerea detectarii MAR](#31---understanding-mar-detection)
    - **[Exercitiul 3 - `test_mar_pattern`](#exercise-3---test_mar_pattern)**
- [4 - Corelatii ale tiparelor de lipsa](#4---missingness-correlations)
    - [4.1 - De ce conteaza corelatiile](#41---why-correlations-matter)
    - **[Exercitiul 4 - `compute_missingness_correlation`](#exercise-4---compute_missingness_correlation)**
- [5 - Clasificarea tipului de lipsa](#5---classifying-missingness-type)
    - **[Exercitiul 5 - `classify_missingness_type`](#exercise-5---classify_missingness_type)**


<a name='imports'></a>
## Importuri

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

In [ ]:
import helper_utils
import unittests

<a name='1---understanding-missingness-patterns'></a>
## 1 - Înțelegerea Tiparelor de Valori Lipsă

<a name='11---generating-sample-data'></a>
### 1.1 - Generarea de Date Exemplu cu Mecanisme Cunoscute

Vom lucra cu trei seturi de date, fiecare ilustrand unul dintre cele trei mecanisme de lipsa:

1. **Date MCAR** — `income` lipseste aleator, independent de toate variabilele (ca o aruncare de moneda)
2. **Date MAR** — `income` are probabilitate mai mare sa lipseasca pentru persoanele *mai in varsta* (lipsa este determinata de `age`, care este observata)
3. **Date MNAR** — `income` are probabilitate mai mare sa lipseasca pentru *venituri mari* (lipsa este determinata de valoarea neobservata in sine)

Ideea-cheie din teorie: singurul mecanism pe care il putem *testa direct* este MCAR, deoarece orice abatere de la o probabilitate constanta a lipsei se va manifesta ca diferente intre mediile grupurilor.


In [ ]:
# Generate datasets with each missingness mechanism
mcar_data = helper_utils.generate_mcar_data(n_samples=500, missing_rate=0.2)
mar_data  = helper_utils.generate_mar_data(n_samples=500, missing_rate=0.3)
mnar_data = helper_utils.generate_mnar_data(n_samples=500, missing_rate=0.3)

print("MCAR Data — missing income:", mcar_data['income'].isnull().sum(),
      f"({mcar_data['income'].isnull().mean()*100:.1f}%)")
print("MAR Data  — missing income:", mar_data['income'].isnull().sum(),
      f"({mar_data['income'].isnull().mean()*100:.1f}%)")
print("MNAR Data — missing income:", mnar_data['income'].isnull().sum(),
      f"({mnar_data['income'].isnull().mean()*100:.1f}%)")

In [ ]:
# MCAR: age distribution should look the same whether income is missing or not
print("MCAR Pattern — age distribution should be similar for both groups")
helper_utils.plot_missingness_vs_values(mcar_data, 'income', 'age')

In [ ]:
# MAR: higher age → more missing income; clear age shift between groups
print("MAR Pattern — older individuals should have more missing income")
helper_utils.plot_missingness_vs_values(mar_data, 'income', 'age')

<a name='exercise-1---create_missingness_indicators'></a>
### **Exercitiul 1 - `create_missingness_indicators`**

**Sarcina ta:**

Implementeaza `create_missingness_indicators` astfel incat sa creeze o variabila indicator binara de lipsa pentru fiecare coloana din DataFrame-ul de intrare.

**Cerinte:**
- Pentru fiecare coloana din `df`, creeaza o coloana corespunzatoare `{col}_missing`
- Indicatorul este `1` daca valoarea lipseste (`NaN`), `0` in caz contrar
- Returneaza un DataFrame *nou* care contine **doar** coloanele indicator

<details>
  <summary><b><font color="green">Indicatii suplimentare de cod (click pentru extindere daca te-ai blocat)</font></b></summary>

**Pasul 1:** `df.isnull()` produce un DataFrame boolean unde `True` = lipsa

**Pasul 2:** `.astype(int)` converteste `True/False` → `1/0`

**Pasul 3:** Redenumeste coloanele: `indicators.columns = [f"{c}_missing" for c in df.columns]`

</details>


In [ ]:
# GRADED FUNCTION: create_missingness_indicators
def create_missingness_indicators(df):
    """
    Create binary indicator variables for each column's missingness.

    Args:
        df (pd.DataFrame): Input DataFrame with potential missing values.

    Returns:
        pd.DataFrame: DataFrame with binary indicators (1=missing, 0=present).
                      Column names are '{original_name}_missing'.
    """
    ### ÎNCEPUT DE COD AICI ###

    # Step 1: Boolean mask for missing values
    missing_mask = None

    # Step 2: Convert boolean to integer
    indicators = None

    # Step 3: Rename columns with '_missing' suffix
    indicators.columns = None

    ### SFÂRȘIT DE COD AICI ###

    return indicators

In [ ]:
test_df = pd.DataFrame({
    'A': [1.0, 2.0, np.nan, 4.0],
    'B': [np.nan, 2.0, 3.0, np.nan],
    'C': [1.0, 2.0, 3.0, 4.0]
})
print("Original DataFrame:")
print(test_df)
print("\nMissingness Indicators:")
print(create_missingness_indicators(test_df))

In [ ]:
# Testează-ți codul!
unittests.exercise_1(create_missingness_indicators)

<a name='2---testing-for-mcar'></a>
## 2 - Testarea MCAR

<a name='21---the-logic-behind-mcar-testing'></a>
### 2.1 - Logica din Spatele Testarii MCAR

Testul MCAR al lui Little (din teorie) verifica daca **mediile variabilelor observate difera intre subgrupurile definite de tiparele de lipsa**. Testul formal de ipoteza este:

- $H_0$: Datele sunt MCAR — mediile nu difera intre grupurile de lipsa
- $H_1$: Datele **nu** sunt MCAR

**Regula de decizie:** Daca $p \leq 0.05$, respingi $H_0$ si concluzionezi ca datele sunt MAR sau MNAR.

Aici implementam o versiune *simplificata* folosind un test t pentru doua esantioane: impartim setul de date in randuri unde o coloana tinta lipseste vs. randuri unde este prezenta, apoi testam daca media unei coloane predictor difera semnificativ intre cele doua grupuri. O diferenta semnificativa ($p < 0.05$) reprezinta evidenta **impotriva** MCAR.


<a name='exercise-2---simplified_mcar_test'></a>
### **Exercitiul 2 - `simplified_mcar_test`**

**Sarcina ta:**

Implementeaza `simplified_mcar_test(df, target_col, predictor_col)` astfel incat sa testeze daca media lui `predictor_col` difera semnificativ intre randurile unde `target_col` lipseste si cele unde este prezent.

**Cerinte:**
- Foloseste `scipy.stats.ttest_ind` pentru testul t cu doua esantioane
- Returneaza un tuplu `(t_statistic, p_value)` — ambele ca valori float
- Un p-value mic ($< 0.05$) inseamna evidenta **impotriva** MCAR

<details>
  <summary><b><font color="green">Indicatii suplimentare de cod (click pentru extindere daca te-ai blocat)</font></b></summary>

**Pasul 1:** Creeaza o masca booleana pentru randurile cu lipsa: `mask = df[target_col].isnull()`

**Pasul 2:** Imparte predictorul in doua grupuri:
```python
group_missing  = df.loc[mask,  predictor_col].dropna()
group_present  = df.loc[~mask, predictor_col].dropna()
```

**Pasul 3:** `stats.ttest_ind(group_missing, group_present)` returneaza un named tuple cu `.statistic` si `.pvalue`

</details>


In [ ]:
# GRADED FUNCTION: simplified_mcar_test
def simplified_mcar_test(df, target_col, predictor_col):
    """
    Test whether the mean of predictor_col differs between rows where target_col is missing vs. present.

    Args:
        df (pd.DataFrame): Input DataFrame.
        target_col (str): Column whose missingness defines the two groups.
        predictor_col (str): Column whose mean is compared between groups.

    Returns:
        tuple: (t_statistic, p_value) — both floats.
    """
    ### ÎNCEPUT DE COD AICI ###

    # Step 1: Boolean mask — True where target_col is missing
    mask = None

    # Step 2: Split predictor into two groups
    group_missing = None
    group_present = None

    # Step 3: Two-sample t-test
    t_stat, p_value = None, None

    ### SFÂRȘIT DE COD AICI ###

    return (t_stat, p_value)

In [ ]:
# MCAR: t-test on age should show NO significant difference
t_mcar, p_mcar = simplified_mcar_test(mcar_data, 'income', 'age')
print(f"MCAR data — t={t_mcar:.3f}, p={p_mcar:.3f}")
print(f"  Interpretation: {'Evidence against MCAR' if p_mcar < 0.05 else 'Consistent with MCAR'}")

# MAR: t-test on age SHOULD show a significant difference
t_mar, p_mar = simplified_mcar_test(mar_data, 'income', 'age')
print(f"\nMAR data  — t={t_mar:.3f}, p={p_mar:.3f}")
print(f"  Interpretation: {'Evidence against MCAR (MAR/MNAR)' if p_mar < 0.05 else 'Consistent with MCAR'}")

In [ ]:
# Testează-ți codul!
unittests.exercise_2(simplified_mcar_test)

<a name='3---testing-for-mar'></a>
## 3 - Testarea MAR

<a name='31---understanding-mar-detection'></a>
### 3.1 - Intelegerea Detectarii MAR

Din teorie: un complement practic al testului lui Little este **corelatia indicatorilor de lipsa**: creezi indicatori binari si testezi asocierea lor cu variabile observate.

Daca un indicator de lipsa (de exemplu, `income_missing = 1` unde venitul lipseste) **se coreleaza cu o variabila complet observata** precum `age`, asta inseamna ca probabilitatea ca `income` sa lipseasca poate fi *prezisa* din ceea ce stii deja. Aceasta este definitia lui MAR: lipsa depinde de date observate, nu de valoarea lipsa in sine.

Cuantificam acest lucru folosind o **corelatie point-biserial** (corelatie Pearson intre un indicator binar si o variabila continua) sau un test t.


<a name='exercise-3---test_mar_pattern'></a>
### **Exercitiul 3 - `test_mar_pattern`**

**Sarcina ta:**

Implementeaza `test_mar_pattern(df, missing_col, predictor_col)` astfel incat sa testeze asocierea dintre un indicator de lipsa si o coloana predictor observata.

**Cerinte:**
- Creeaza un indicator binar: `1` acolo unde `missing_col` este `NaN`, `0` in rest
- Calculeaza **corelatia Pearson** dintre indicator si `predictor_col`
- Returneaza un tuplu `(correlation, p_value)`

<details>
  <summary><b><font color="green">Indicatii suplimentare de cod (click pentru extindere daca te-ai blocat)</font></b></summary>

**Pasul 1:** Creeaza indicatorul: `indicator = df[missing_col].isnull().astype(int)`

**Pasul 2:** Elimina randurile unde `predictor_col` este si el `NaN` (pentru a calcula o corelatie valida)

**Pasul 3:** `stats.pearsonr(indicator_aligned, predictor_aligned)` returneaza `(r, p_value)`

</details>


In [ ]:
# GRADED FUNCTION: test_mar_pattern
def test_mar_pattern(df, missing_col, predictor_col):
    """
    Test whether missingness in missing_col is correlated with observed values of predictor_col.

    Args:
        df (pd.DataFrame): Input DataFrame.
        missing_col (str): Column whose missingness indicator to create.
        predictor_col (str): Fully observed column to correlate against.

    Returns:
        tuple: (correlation, p_value) — both floats.
    """
    ### ÎNCEPUT DE COD AICI ###

    # Step 1: Create binary missingness indicator
    indicator = None

    # Step 2: Align — drop rows where predictor_col is NaN
    valid_mask = None
    indicator_aligned = None
    predictor_aligned = None

    # Step 3: Compute Pearson correlation
    correlation, p_value = None, None

    ### SFÂRȘIT DE COD AICI ###

    return (correlation, p_value)

In [ ]:
# MCAR: no correlation expected
r_mcar, p_mcar = test_mar_pattern(mcar_data, 'income', 'age')
print(f"MCAR — r={r_mcar:.3f}, p={p_mcar:.3f}: {'MAR signal detected' if p_mcar < 0.05 else 'No MAR signal'}")

# MAR: strong correlation expected
r_mar, p_mar = test_mar_pattern(mar_data, 'income', 'age')
print(f"MAR  — r={r_mar:.3f}, p={p_mar:.3f}: {'MAR signal detected' if p_mar < 0.05 else 'No MAR signal'}")

In [ ]:
# Testează-ți codul!
unittests.exercise_3(test_mar_pattern)

<a name='4---missingness-correlations'></a>
## 4 - Corelații de Lipsă

<a name='41---why-correlations-matter'></a>
### 4.1 - De ce Conteaza Corelatiile

Din teorie: heatmap-ul `missingno` afiseaza **corelatii perechi ale lipsei**:
- **+1** — cele doua coloane lipsesc *mereu* pe aceleasi randuri (cauza comuna in amonte)
- **-1** — coloanele *nu* lipsesc niciodata impreuna (cai mutual exclusive)
- **0** — lipsa dintr-o coloana nu ofera informatie despre cealalta

Acest lucru identifica grupuri de trasaturi care ar trebui imputate impreuna (KNN sau MICE), nu independent pe coloane.


<a name='exercise-4---compute_missingness_correlation'></a>
### **Exercitiul 4 - `compute_missingness_correlation`**

**Sarcina ta:**

Implementeaza `compute_missingness_correlation(df)` astfel incat sa calculeze matricea de corelatie a indicatorilor de lipsa pentru toate coloanele care au cel putin o valoare lipsa.

**Cerinte:**
- Include doar coloanele care au cel putin o valoare lipsa
- Creeaza indicatori binari si calculeaza matricea de corelatie Pearson
- Returneaza un `pd.DataFrame` (matrice de corelatie, valori intre −1 si +1)

<details>
  <summary><b><font color="green">Indicatii suplimentare de cod (click pentru extindere daca te-ai blocat)</font></b></summary>

**Pasul 1:** `cols_with_missing = df.columns[df.isnull().any()].tolist()`

**Pasul 2:** Creeaza indicatorii pentru acele coloane: `df[cols_with_missing].isnull().astype(int)`

**Pasul 3:** `.corr()` aplicat pe DataFrame-ul de indicatori returneaza matricea de corelatie

</details>


In [ ]:
# GRADED FUNCTION: compute_missingness_correlation
def compute_missingness_correlation(df):
    """
    Compute pairwise correlations between missingness indicators.

    Args:
        df (pd.DataFrame): Input DataFrame.

    Returns:
        pd.DataFrame: Correlation matrix of missingness indicators.
    """
    ### ÎNCEPUT DE COD AICI ###

    # Step 1: Identify columns with at least one missing value
    cols_with_missing = None

    # Step 2: Create binary indicators
    indicators = None

    # Step 3: Compute correlation matrix
    corr_matrix = None

    ### SFÂRȘIT DE COD AICI ###

    return corr_matrix

In [ ]:
# Check on MCAR data — low correlations expected (random missingness)
corr_mcar = compute_missingness_correlation(mcar_data)
print("Missingness correlation matrix (MCAR data):")
print(corr_mcar.round(3))

In [ ]:
# Testează-ți codul!
unittests.exercise_4(compute_missingness_correlation)

<a name='5---classifying-missingness-type'></a>
## 5 - Clasificarea Tipului de Lipsa

Combinand testele de mai sus, putem construi un clasificator euristic care eticheteaza lipsa unei coloane drept probabil MCAR, MAR sau MNAR. Logica de clasificare urmeaza cadrul decizional din teorie:

1. Ruleaza testul MCAR simplificat pe coloanele predictor
2. Daca niciun predictor nu produce un p-value semnificativ → probabil **MCAR**
3. Daca cel putin un predictor arata o corelatie puternica → probabil **MAR**
4. Daca valorile decalate/corelate ale coloanei sugereaza autodependenta → marcheaza drept posibil **MNAR**


<a name='exercise-5---classify_missingness_type'></a>
### **Exercitiul 5 - `classify_missingness_type`**

**Sarcina ta:**

Implementeaza `classify_missingness_type(df, target_col, significance_level=0.05)` astfel incat sa returneze una dintre etichetele: `'MCAR'`, `'MAR'` sau `'MNAR'`.

**Cerinte:**
- Testeaza fiecare coloana numerica din `df` (cu exceptia lui `target_col`) ca predictor potential
- Foloseste `simplified_mcar_test` pentru diferente intre mediile grupurilor
- Foloseste `test_mar_pattern` pentru semnalul MAR bazat pe corelatie
- Returneaza `'MCAR'` daca niciun predictor nu este semnificativ; `'MAR'` daca cel putin un predictor se coreleaza; `'MNAR'` daca dovezile sunt ambigue / apare autocorelatie ridicata

<details>
  <summary><b><font color="green">Indicatii suplimentare de cod (click pentru extindere daca te-ai blocat)</font></b></summary>

**Pasul 1:** Obtine coloanele numerice, excluzand `target_col`

**Pasul 2:** Pentru fiecare predictor, ruleaza atat `simplified_mcar_test`, cat si `test_mar_pattern`

**Pasul 3:** Numara rezultatele semnificative. Daca vreun p-value al corelatiei este sub prag → `'MAR'`. Daca niciun p-value nu este semnificativ → `'MCAR'`. In rest → `'MNAR'`.

</details>


In [ ]:
# GRADED FUNCTION: classify_missingness_type
def classify_missingness_type(df, target_col, significance_level=0.05):
    """
    Classify the likely missingness mechanism for a column.

    Args:
        df (pd.DataFrame): Input DataFrame.
        target_col (str): Column to classify.
        significance_level (float): P-value threshold. Default 0.05.

    Returns:
        str: One of 'MCAR', 'MAR', or 'MNAR'.
    """
    ### ÎNCEPUT DE COD AICI ###

    # Step 1: Get numeric predictor columns (excluding target)
    numeric_cols = df.select_dtypes(include='number').columns.tolist()
    predictors = [c for c in numeric_cols if c != target_col]

    mar_signals = 0
    mcar_consistent = 0

    for predictor in predictors:
        # Skip if predictor has no variance after dropping NaN
        if df[predictor].dropna().std() == 0:
            continue

        # Step 2a: Group mean difference test
        try:
            _, p_ttest = simplified_mcar_test(df, target_col, predictor)
        except Exception:
            continue

        # Step 2b: Correlation test
        try:
            _, p_corr = test_mar_pattern(df, target_col, predictor)
        except Exception:
            continue

        if p_ttest < significance_level or p_corr < significance_level:
            mar_signals += 1
        else:
            mcar_consistent += 1

    # Step 3: Classification rule
    if mar_signals == 0:
        classification = None  # No predictor triggers significance -> MCAR
    elif mar_signals > 0 and mar_signals <= len(predictors) // 2:
        classification = None  # Some predictors significant -> MAR
    else:
        classification = None  # Widespread significance or ambiguous -> MNAR

    ### SFÂRȘIT DE COD AICI ###

    return classification

In [ ]:
print("MCAR data classification:", classify_missingness_type(mcar_data, 'income'))
print("MAR data  classification:", classify_missingness_type(mar_data,  'income'))
print("MNAR data classification:", classify_missingness_type(mnar_data, 'income'))

In [ ]:
# Testează-ți codul!
unittests.exercise_5(classify_missingness_type)